In [1]:
# Cell 0a - Install wheel first
!pip install -q wheel
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers trl peft accelerate bitsandbytes
!pip install -q flash-attn --no-build-isolation

In [2]:
# Cell 0b - Install everything else
!pip install -q transformers trl peft accelerate bitsandbytes flash-attn dotenv --no-build-isolation

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

/home/ubuntu/DiskUsEast1/finetuning_evaluation/venv_test1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [4]:
# Load HF token from .env file
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env from project root
env_path = Path('/home/ubuntu/finetuning_evaluation/.env')
load_dotenv(dotenv_path=env_path)
hf_token = os.environ.get('HF_TOKEN', None)

if hf_token is None:
    print("❌ Error: HF_TOKEN not found in .env file")
    print(f"   Expected location: {env_path}")
    exit(1)
else:
    print(f"✅ HF_TOKEN loaded successfully from .env")

✅ HF_TOKEN loaded successfully from .env


In [5]:
# ===== LOAD MODEL (SMALLER FOR LOCAL) =====
# Use Llama-3.2-1B instead of 8B (fits in RTX A6000)
model_id = "meta-llama/Llama-3.1-8B"  # ✅ CHANGED: 8B vs 1B

print("\n📦 Loading model with Flash Attention 2...")

try:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="flash_attention_2",  # ✅ Testing this
        token=hf_token  # Use the token from environment
    )
    print("✅ Model loaded successfully with Flash Attention 2")
except Exception as e:
    print(f"❌ Model loading failed: {e}")
    exit(1)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token) # Use the token for tokenizer as well
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = 512  # Short for testing


📦 Loading model with Flash Attention 2...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.39it/s]


✅ Model loaded successfully with Flash Attention 2


In [6]:
# ===== APPLY LORA =====
print("\n🔧 Applying LoRA...")
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,  # ✅ CHANGED: 16 vs 8 (match production)
    lora_alpha=16,  # ✅ CHANGED: 16 vs 8
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],  # ✅ CHANGED: All 7 modules (match production)
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


🔧 Applying LoRA...
trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [7]:
# ===== CREATE DUMMY DATASET =====
print("\n📊 Creating dummy dataset...")
dummy_data = {
    "prompt": ["Hello, how are you?"] * 8,
    "chosen": ["I'm doing great, thank you!"] * 8,
    "rejected": ["I don't know."] * 8,
}
dataset = Dataset.from_dict(dummy_data)


📊 Creating dummy dataset...


In [8]:
# ===== TEST 1: WITHOUT ref_model_dtype (expected to fail?) =====
print("\n" + "="*80)
print("TEST 1: DPOConfig WITHOUT ref_model_dtype (baseline)")
print("="*80)

training_args_v1 = DPOConfig(
    output_dir="./test_output_v1",
    beta=0.1,
    per_device_train_batch_size=1,
    max_steps=5,  # Just 5 steps for testing
    learning_rate=2e-5,
    logging_steps=1,
    bf16=True,
    # NO ref_model_dtype specified
    report_to="none", # Skip wandb
)

try:
    trainer_v1 = DPOTrainer(
        model=model,
        ref_model=None,  # DPOTrainer creates reference model internally
        args=training_args_v1,
        train_dataset=dataset,
        processing_class=tokenizer,
    )
    print("✅ DPOTrainer initialized successfully")

    print("\n🏃 Running 5 training steps...")
    trainer_v1.train()
    print("✅ TEST 1 PASSED: Training completed without dtype errors")

except RuntimeError as e:
    if "dtype" in str(e).lower() or "bfloat16" in str(e).lower():
        print(f"❌ TEST 1 FAILED: Dtype error (expected)")
        print(f"   Error: {e}")
    else:
        print(f"❌ TEST 1 FAILED: Unexpected error")
        print(f"   Error: {e}")

except Exception as e:
    print(f"❌ TEST 1 FAILED: {e}")


TEST 1: DPOConfig WITHOUT ref_model_dtype (baseline)


Tokenizing train dataset: 100%|██████████| 8/8 [00:00<00:00, 467.23 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


✅ DPOTrainer initialized successfully

🏃 Running 5 training steps...


Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.
/home/ubuntu/DiskUsEast1/finetuning_evaluation/venv_test1/lib/python3.10/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss
1,0.693100
2,0.689700
3,0.666800
4,0.670800
5,0.655900


✅ TEST 1 PASSED: Training completed without dtype errors


In [9]:
# ===== TEST 2: WITH ref_model_dtype (testing fix) =====
print("\n" + "="*80)
print("TEST 2: DPOConfig WITH ref_model_dtype=torch.bfloat16 (proposed fix)")
print("="*80)

training_args_v2 = DPOConfig(
    output_dir="./test_output_v2",
    beta=0.1,
    per_device_train_batch_size=1,
    max_steps=5,
    learning_rate=2e-5,
    logging_steps=1,
    bf16=True,
    # ✅ TESTING THIS FIX - Removed ref_model_dtype as it's not supported
    ref_model_dtype=torch.bfloat16,  # Force reference model to use BF16
    # TO AVOID: TypeError: DPOConfig.__init__() got an unexpected keyword argument 'ref_model_dtype'
    report_to="none", # Skip wandb
)

try:

    model2 = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="flash_attention_2",
        token=hf_token # Use HF token from secrets
    )
    model2 = prepare_model_for_kbit_training(model2, use_gradient_checkpointing=True)
    model2 = get_peft_model(model2, lora_config)

    trainer_v2 = DPOTrainer(
        model=model2,
        ref_model=None,
        args=training_args_v2,
        train_dataset=dataset,
        processing_class=tokenizer,
    )
    print("✅ DPOTrainer initialized successfully")

    print("\n🏃 Running 5 training steps...")
    trainer_v2.train()
    print("✅ TEST 2 PASSED: Training completed with ref_model_dtype fix")

except RuntimeError as e:
    if "dtype" in str(e).lower():
        print(f"❌ TEST 2 FAILED: Dtype error persists (fix didn't work)")
        print(f"   Error: {e}")
    else:
        print(f"❌ TEST 2 FAILED: Unexpected error")
        print(f"   Error: {e}")

except Exception as e:
    print(f"❌ TEST 2 FAILED: {e}")


TEST 2: DPOConfig WITH ref_model_dtype=torch.bfloat16 (proposed fix)


TypeError: DPOConfig.__init__() got an unexpected keyword argument 'ref_model_dtype'

In [10]:
# ===== SUMMARY =====
print("\n" + "="*80)
print("📊 TEST SUMMARY")
print("="*80)
print("Test 1 (no ref_model_dtype): Check output above")
print("Test 2 (with ref_model_dtype): Check output above")
print("\nIf both tests pass, Flash Attention 2 works without dtype fix.")
print("If Test 1 fails but Test 2 passes, the fix works!")
print("If both fail, Flash Attention 2 is incompatible with CITA/DPO.")
print("="*80)

# ===== MEMORY REPORT =====
if torch.cuda.is_available():
    print(f"\n📊 GPU Memory Usage:")
    print(f"  Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"  Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


📊 TEST SUMMARY
Test 1 (no ref_model_dtype): Check output above
Test 2 (with ref_model_dtype): Check output above

If both tests pass, Flash Attention 2 works without dtype fix.
If Test 1 fails but Test 2 passes, the fix works!
If both fail, Flash Attention 2 is incompatible with CITA/DPO.

📊 GPU Memory Usage:
  Allocated: 30.40 GB
  Reserved:  35.62 GB
